# 00 — Product source probes (T-011/S2)

Analyses the bounded read-only probe run produced by `ml/probes/run_probes.py` (protocol: `docs/PRODUCT_DATA_API_EVALUATION.md`). Inputs are presence flags and hashes only; no raw provider payloads are stored. Question: does combining providers through the router (design §2) improve field completeness over any single provider?

In [1]:
import json, platform
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'notebooks/results/00_source_probes'
obs = pd.read_csv(OUT / 'observations.csv', dtype={'value': str})
print(platform.python_version(), pd.__version__, len(obs), 'observations')
obs[['corpus_id','provider','http_status','outcome','elapsed_ms','product_type']]

3.13.0 3.0.6 26 observations


,corpus_id,provider,http_status,outcome,elapsed_ms,product_type
0,P01,open_food_facts,200,KNOWN,201,food
1,P02,open_food_facts,200,KNOWN,141,food
2,P03,open_food_facts,200,KNOWN,152,food
3,P04,open_food_facts,200,KNOWN,167,food
4,P05,open_food_facts,200,KNOWN,146,food
5,P06,open_food_facts,200,KNOWN,128,food
6,P07,open_food_facts,404,UNKNOWN,177,NaN
7,P08,open_food_facts,404,UNKNOWN,153,NaN
8,P09,open_food_facts,404,UNKNOWN,155,NaN
9,P10,open_food_facts,404,UNKNOWN,138,NaN


## Outcome classes per provider
U01 is the not-found control and is excluded from coverage.

In [2]:
cov = obs[obs.corpus_id != 'U01']
outcomes = pd.crosstab(cov.provider, cov.outcome)
control = obs[obs.corpus_id == 'U01'][['provider','outcome']]
display(outcomes); display(control)

outcome,KNOWN,UNAVAILABLE,UNKNOWN
provider,,,
open_food_facts,7,0,5
upcitemdb,8,1,3


,provider,outcome
12,open_food_facts,KNOWN
25,upcitemdb,KNOWN


## Field completeness: single providers vs router merge
Merged = a field is present if *any* provider returned it for that product (field-level merge).

In [3]:
FIELDS = ['has_name','has_brand','has_category','has_description']
flags = cov.copy()
for f in FIELDS: flags[f] = flags[f].fillna(False).astype(str).eq('True')
per = flags.pivot_table(index='corpus_id', columns='provider', values=FIELDS, aggfunc='first')
rows = {}
for p in flags.provider.unique():
    rows[p] = {f: per[(f, p)].mean() for f in FIELDS}
rows['router_merge'] = {f: per[f].any(axis=1).mean() for f in FIELDS}
completeness = pd.DataFrame(rows).T.rename(columns=lambda c: c.replace('has_',''))
completeness['known_rate'] = [ (cov[cov.provider==p].outcome=='KNOWN').mean() for p in flags.provider.unique()] + [per['has_name'].any(axis=1).mean()]
completeness.round(3)

,name,brand,category,description,known_rate
open_food_facts,0.583,0.583,0.583,0.083,0.583
upcitemdb,0.667,0.667,0.583,0.583,0.667
router_merge,1.000,1.000,1.000,0.667,1.000


In [4]:
ax = completeness.plot.bar(figsize=(8,4), rot=0, ylim=(0,1.05))
ax.set_ylabel('share of 12 corpus products'); ax.set_title('Field completeness by source (n=12)')
plt.tight_layout(); plt.savefig(OUT / 'completeness.png', dpi=150); plt.show()

/tmp/ipykernel_148844/3113614253.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'completeness.png', dpi=150); plt.show()


## Which provider covers which product

In [5]:
display(per['has_name'].replace({True:'✓', False:'—'}))

provider,open_food_facts,upcitemdb
corpus_id,,
P01,✓,✓
P02,✓,—
P03,✓,—
P04,✓,—
P05,✓,—
P06,✓,✓
P07,—,✓
P08,—,✓
P09,—,✓


## Latency (single sequential calls, desktop, not phone timing)

In [6]:
lat = obs.groupby('provider').elapsed_ms.describe()[['count','mean','50%','max']]
lat.round(0)

,count,mean,50%,max
provider,,,,
open_food_facts,13.0,157.0,153.0,201.0
upcitemdb,13.0,523.0,523.0,555.0


## Identity handling (EAN-8 / UPC-E / UPC-A canonicalisation)

In [7]:
display(obs[obs.corpus_id.isin(['P01','P02','P06'])][['corpus_id','value','format','provider','outcome','returned_codes']])

,corpus_id,value,format,provider,outcome,returned_codes
0,P01,42070047,EAN_8,open_food_facts,KNOWN,42070047
1,P02,04963406,UPC_E,open_food_facts,KNOWN,04963406
5,P06,069000019832,UPC_A,open_food_facts,KNOWN,0069000019832
13,P01,42070047,EAN_8,upcitemdb,KNOWN,0000042070047|000042070047
14,P02,04963406,UPC_E,upcitemdb,UNAVAILABLE,NaN
18,P06,069000019832,UPC_A,upcitemdb,KNOWN,0069000019832|069000019832


## Export

In [8]:
summary = {
  'run_rows': len(obs), 'calls_per_provider': obs.provider.value_counts().to_dict(),
  'completeness': completeness.round(4).to_dict(orient='index'),
  'latency_ms': lat.round(1).to_dict(orient='index'),
  'control_U01': control.set_index('provider').outcome.to_dict(),
}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
completeness.round(4).to_csv(OUT / 'completeness.csv'); print(json.dumps(summary, indent=1))

{
 "run_rows": 26,
 "calls_per_provider": {
  "open_food_facts": 13,
  "upcitemdb": 13
 },
 "completeness": {
  "open_food_facts": {
   "name": 0.5833,
   "brand": 0.5833,
   "category": 0.5833,
   "description": 0.0833,
   "known_rate": 0.5833
  },
  "upcitemdb": {
   "name": 0.6667,
   "brand": 0.6667,
   "category": 0.5833,
   "description": 0.5833,
   "known_rate": 0.6667
  },
  "router_merge": {
   "name": 1.0,
   "brand": 1.0,
   "category": 1.0,
   "description": 0.6667,
   "known_rate": 1.0
  }
 },
 "latency_ms": {
  "open_food_facts": {
   "count": 13.0,
   "mean": 157.0,
   "50%": 153.0,
   "max": 201.0
  },
  "upcitemdb": {
   "count": 13.0,
   "mean": 523.0,
   "50%": 523.0,
   "max": 555.0
  }
 },
 "control_U01": {
  "open_food_facts": "KNOWN",
  "upcitemdb": "KNOWN"
 }
}


## Limitations
12 products is a qualitative coverage check, not a statistical estimate. Corpus items P07–P12 were sourced from UPCitemdb's own public catalog, which biases toward it; P02–P06 come from OFF documentation, which biases toward OFF. The merge result is the meaningful comparison: the two sources are complementary.